# 04 Deep dive: your own matrices, and solving somewhere else

The 01 notebooks let a modelling framework build the problem. This one builds it by hand, with
numpy and scipy, so you see the interface for what it is. Two situations it is for:

1. You build the matrices yourself: your model generator emits sparse matrices, or you are wiring
   PIPS-IPM++ into a tool with no supported front end.
2. You cannot solve where you build: the model is assembled on a laptop and has to run on a
   cluster, or a run has to be reproducible months later.

| sections | run on |
|---|---|
| 1 to 4: build a `StructuredProblem`, classify its rows, derive an annotation, write it to files | every machine (the hypergraph cell needs mt-KaHyPar) |
| 5 to 6: solve it, then block by block from the files with MPI | Linux or WSL with the `solver` extra |
| 7 to 9: read the solution back, options, processes | every machine, once a solution exists |

In [ ]:
import workshop

workshop.check()

## 1. `StructuredProblem`

Everything in the package is built on one dataclass:

| field | shape | meaning |
|---|---|---|
| `n_blocks` | | number of leaf blocks; the root is not counted |
| `var_block` | `(n,)` int | block of each variable, `0` = root, `1..n_blocks` = leaf |
| `c` | `(n,)` | objective coefficients, always a minimisation |
| `xlow`, `xupp` | `(n,)` | variable bounds, `-inf` or `inf` for none |
| `A_eq`, `b_eq` | `(m_eq, n)` CSR | equality rows, `A_eq x == b_eq` |
| `A_ineq` | `(m_ineq, n)` CSR | inequality rows |
| `ineq_low`, `ineq_upp` | `(m_ineq,)` | two-sided: `ineq_low <= A_ineq x <= ineq_upp`, `-inf` or `inf` for an open side |
| `objconst` | scalar | added to the objective, so a reported value matches your own |

- Ranges are free. `1 <= a x <= 5` is one row, not two: use it for reserve margins, ramping
  limits, storage windows.
- Minimisation only. For a maximisation negate `c`, and negate the objective and the duals on the
  way back. Every framework does this for you; building the problem yourself, it is yours.

The sample model from 00, with 24 hours and 4 blocks, built by hand. Names are kept for later.

In [ ]:
import numpy as np
import scipy.sparse as sp

from sample_data import TECHS, budget, profiles

T, N_BLOCKS = 24, 4
HOURS = T // N_BLOCKS                          # hours per leaf block
techs = list(TECHS)
demand, avail = profiles(T)
invcost, varcost, emission = (np.array([TECHS[g][k] for g in techs]) for k in range(3))
availability = np.stack([avail[TECHS[g][3]] for g in techs])

n_techs = len(techs)
n_vars = n_techs + n_techs * T


def gen_ix(g, t):
    # column of gen[g, t]; columns 0 .. n_techs-1 are the capacities
    return n_techs + g * T + t


names = [f"cap({g})" for g in techs] + [f"gen({g},{t})" for g in techs for t in range(T)]
print(f"{n_vars} variables: {n_techs} capacities + {n_techs * T} dispatch")

### The annotation: this is the whole interface

In [ ]:
var_block = np.zeros(n_vars, dtype=np.int64)   # 0 = root, and the capacities stay there
for g in range(n_techs):
    for t in range(T):
        var_block[gen_ix(g, t)] = 1 + t // HOURS

print("capacities  ->", var_block[:n_techs])
print("gen[wind,:] ->", var_block[n_techs:n_techs + T])

### The matrices

In [ ]:
c = np.zeros(n_vars)
c[:n_techs] = invcost
for g in range(n_techs):
    for t in range(T):
        c[gen_ix(g, t)] = varcost[g]

# balance:  sum_g gen[g, t] == demand[t]
rows, cols, vals = [], [], []
for t in range(T):
    for g in range(n_techs):
        rows.append(t); cols.append(gen_ix(g, t)); vals.append(1.0)
A_eq = sp.csr_array((vals, (rows, cols)), shape=(T, n_vars))

# limit:  gen[g, t] - avail[g, t] * cap[g] <= 0,  then the carbon budget as the last row
rows, cols, vals = [], [], []
r = 0
for g in range(n_techs):
    for t in range(T):
        rows += [r, r]; cols += [gen_ix(g, t), g]; vals += [1.0, -availability[g, t]]
        r += 1
for g in range(n_techs):
    for t in range(T):
        if emission[g]:
            rows.append(r); cols.append(gen_ix(g, t)); vals.append(emission[g])
A_ineq = sp.csr_array((vals, (rows, cols)), shape=(r + 1, n_vars))
ineq_low = np.full(r + 1, -np.inf)                         # no lower side
ineq_upp = np.concatenate([np.zeros(r), [budget(T)]])

row_names = ([f"balance({t})" for t in range(T)]
             + [f"limit({g},{t})" for g in techs for t in range(T)] + ["carbon"])
print(f"{A_eq.shape[0]} equalities, {A_ineq.shape[0]} inequalities, "
      f"{A_eq.nnz + A_ineq.nnz} nonzeros")

In [ ]:
import pipsipmpp

problem = pipsipmpp.StructuredProblem(
    n_blocks=N_BLOCKS, var_block=var_block, c=c,
    xlow=np.zeros(n_vars), xupp=np.full(n_vars, np.inf),
    A_eq=A_eq, b_eq=demand, A_ineq=A_ineq, ineq_low=ineq_low, ineq_upp=ineq_upp,
)
problem.validate()          # catches shape and format mistakes before the solver sees them
print(f"{problem.n_vars} variables, {problem.n_blocks} leaves: valid")

<details><summary>💡 Hint: <code>validate()</code> raised an <code>AssertionError</code></summary>

It checks shapes (every vector the right length), formats (both matrices CSR) and that every
`var_block` lies in `0 ... n_blocks`. The usual culprits: a `csc` or `coo` matrix, converted with
`.tocsr()`, or `n_blocks` set lower than the largest block number you used.
</details>

## 2. What the annotation does to the rows

You annotated the columns. The rows classify themselves. Check that it matches what you expected;
this is the single best habit to build.

In [ ]:
from collections import Counter


def row_kinds(A, var_block):
    # 'root' if a row touches root columns only, 'leaf k' for one leaf, 'linking' for several
    kinds = []
    for i in range(A.shape[0]):
        leaves = set(var_block[A.indices[A.indptr[i]:A.indptr[i + 1]]].tolist()) - {0}
        kinds.append("root" if not leaves else
                     f"leaf {leaves.pop()}" if len(leaves) == 1 else "linking")
    return Counter(kinds)


print("equalities  ", dict(sorted(row_kinds(A_eq, var_block).items())))
print("inequalities", dict(sorted(row_kinds(A_ineq, var_block).items())))

Exactly one linking row, the carbon budget, and everything else local to a leaf.

Notice what the capacity limit `gen[g,t] <= avail * cap[g]` became: local to its leaf, although
it uses the root column `cap[g]`. Root columns do not make a row linking; they sit in the left
border, which every leaf may use. Only a row that touches two different leaves links.

## 3. Let the annotation be derived for you

You do not have to work out `var_block` yourself. `pipsipmpp.annotate()` hands the matrix to
pipstools and returns the same problem with the annotation filled in: the in-memory version of
what 03 did on a file. Start from a problem with no structure at all, everything in the root:

In [ ]:
flat = pipsipmpp.StructuredProblem(
    n_blocks=0, var_block=np.zeros(n_vars, dtype=np.int64), c=c,
    xlow=problem.xlow, xupp=problem.xupp, A_eq=A_eq, b_eq=demand,
    A_ineq=A_ineq, ineq_low=ineq_low, ineq_upp=ineq_upp,
)
by_regex = pipsipmpp.annotate(flat, N_BLOCKS, method="regex", regex=r",(\d+)\)$", names=names)

print("variables per block, root first")
print("  by hand ", np.bincount(problem.var_block).tolist())
print("  by regex", np.bincount(by_regex.var_block).tolist())
print("identical:", bool((by_regex.var_block == problem.var_block).all()))

The regex `,(\d+)\)$` captures the hour from `gen(wind,17)`; `cap(wind)` matches nothing and stays
in the root. It reproduces the hand-written annotation exactly.

The hypergraph method needs mt-KaHyPar, which has no Windows build:

In [ ]:
%%requires mtkahypar
by_graph = pipsipmpp.annotate(flat, N_BLOCKS, method="hypergraph")

for label, p in (("by hand", problem), ("by regex", by_regex), ("hypergraph", by_graph)):
    kinds = row_kinds(p.A_ineq, p.var_block) + row_kinds(p.A_eq, p.var_block)
    print(f"{label:10s} root columns {int((p.var_block == 0).sum()):3d}   "
          f"linking rows {kinds['linking']:3d}")

On a 50-variable problem there is little for a partitioner to work with, and it finds something
different from the obvious split. On a model with a quarter of a million columns (03) it can do
very well. The lesson is the same as in 03: check an annotation, however you got it.

## 4. Write it to files

This is what makes PIPS-IPM++ practical on real models. The annotated problem goes to parquet,
moves anywhere, and is solved without the tool that produced it. Writing needs neither MPI nor
the solver.

| layout | files | use it when |
|---|---|---|
| `monolithic` | one set for the whole problem | it fits comfortably in one process; simplest to move around, and pipstools can read it |
| `distributed` | one set per block | every process reads only its own blocks, so the whole problem is never held anywhere |

In [ ]:
from pathlib import Path

Path("data/deep").mkdir(parents=True, exist_ok=True)
labels = {"cols": names, "rows": row_names}

mono = pipsipmpp.write_problem(problem, "data/deep_mono", layout="monolithic", names=labels)
dist = pipsipmpp.write_problem(problem, "data/deep/model", layout="distributed", names=labels)
print(*sorted(f.name for f in Path("data/deep").iterdir()), sep="\n  ")

workshop.visualize(mono)

`model.manifest.json` ties the distributed set together; `model0.*` is the root, `model1.*`
onwards one set per leaf. The distributed set gets a folder of its own, because the solution is
written next to it into a folder called `solution`. The monolithic set is what `pipstools` and `workshop.visualize` read: the same
picture as in 01.

<details><summary>💡 Hint: why are the equalities and inequalities stored in a different order?</summary>

The format stores every equality row first and then every inequality row, per block, because that
is how PIPS-IPM++ wants them. `names={"rows": ...}` takes your rows in your order (equalities first,
as here), and the writer rearranges them with the matrix, so names and rows stay together.
</details>

## 5. Solve it

From here on the cells need the solver: Linux or WSL, with JupyterLab started through
`uv run --extra solver jupyter lab` or `pixi run -e solver lab`. On other machines they are
skipped.

In the notebook, on one process:

In [ ]:
%%requires solver
result = pipsipmpp.solve(problem, options={"SILENT": True})

In [ ]:
%%requires solver
print(result.status.name, "| optimal:", result.status.is_optimal)
print(f"objective {result.objective:.6f} in {result.iterations} iterations")
print("capacities:", dict(zip(techs, result.primal[:n_techs].round(4).tolist())))
print("carbon dual:", round(float(result.dual_ineq[-1]), 4))

`2.744656`, the objective every 01 notebook found for this model.

## 6. Solve a dataset, block by block

`solve_dataset` reads the distributed files directly on every process, each loading only the
blocks it owns. It runs as a script under MPI:

In [ ]:
%%writefile solve_dataset.py
import pipsipmpp
from mpi4py import MPI

comm = MPI.COMM_WORLD
result = pipsipmpp.solve_dataset("data/deep/model", options={"SILENT": True},
                                 write_solution=True)       # leaves the answer next to it
if comm.Get_rank() == 0:
    print(f"ranks={comm.Get_size()} {result.status.name} objective={result.objective:.6f}")

In [ ]:
%%requires solver
for ranks in (1, 4):
    workshop.mpirun("solve_dataset.py", ranks, show="ranks=")

The same objective on one process and on four. The iteration count may differ by one or two: the
arithmetic is split differently, so it is not bit-for-bit identical.

<details><summary>💡 Hint: what is MPI, and do I have to install it?</summary>

MPI is the standard way to start one program as several cooperating processes that exchange data.
`workshop.mpirun("script.py", 4)` runs `mpiexec -n 4 python script.py` for you. The `solver` extra
installs an MPI runtime into the environment, so there is nothing to install separately.
</details>

## 7. Read the solution back

`write_solution=True` left the answer beside the problem. Reading it needs neither MPI nor the
compiled solver, only the `pipsipmpp` package, so the machine that built the model can read the
result of a run that happened on a cluster, whatever its operating system.

In [ ]:
if Path("data/deep/solution/manifest.json").exists():
    solution = pipsipmpp.read_solution("data/deep/model")
    print(solution.status.name, "| objective", round(solution.objective, 6),
          "| iterations", solution.iterations)
    for name, value in zip(names[:n_techs], solution.primal[:n_techs]):
        print(f"  {name:10s} {value:.4f}")
else:
    print("no solution in data/deep yet: section 6 writes one on Linux or WSL")

The whole workflow:

```
 your generator --> StructuredProblem --> write_problem(..., layout="distributed")
                                                   |  copy the files to the cluster
                          mpiexec -n N python solve_dataset.py
                                                   |  copy the solution back
 your analysis  <-- read_solution(...) <-----------+
```

## 8. Options, and a reproducible run

Options can live in a file, the right place for anything you want to repeat:

```
# tuned.opt: one NAME value per line; # and // start a comment
LINEAR_LEAF_SOLVER   mumps
SCALER               geo_equi
```

```python
pipsipmpp.solve(problem, comm, options_file="tuned.opt", options={"SILENT": True})
```

`options` overrides the file. Only process 0 has to supply either; both are broadcast.

Three things might be useful for debugging and filing issues: the options, the `solver_git_hash` 
from `libpips_info()`, and the annotation. The first two are a line each; the third is why
`write_problem` exists.

In [ ]:
%%requires solver
info = pipsipmpp.libpips_info()
print("solver", info["solver_version"], "git", info["solver_git_hash"])
print("blocks", problem.n_blocks, "| root variables", int((problem.var_block == 0).sum()))

## 9. How blocks map onto processes

Useful when sizing a cluster job, and it runs everywhere:

In [ ]:
for ranks in (1, 2, 4, 8):
    try:
        print(f"{ranks} process(es): leaf -> process",
              pipsipmpp.block_owner(problem.n_blocks, ranks).tolist())
    except AssertionError as err:
        print(f"{ranks} process(es): {err}")

More processes than leaves is refused: the extra ones would have nothing to own. So plan at least
as many leaf blocks as processes, and few enough that the borders stay small.

## 10. Now your own model

The smallest possible integration:

```python
problem = pipsipmpp.StructuredProblem(n_blocks=..., var_block=..., c=..., xlow=..., xupp=...,
                                      A_eq=..., b_eq=..., A_ineq=..., ineq_low=..., ineq_upp=...)
problem.validate()
pipsipmpp.write_problem(problem, "data/mine", layout="monolithic", names=...)
workshop.visualize("data/mine")
result = pipsipmpp.solve(problem, comm)          # on Linux
```

A checklist for wiring in a generator of your own:

1. CSR, `float64`, `int64`. `validate()` checks shapes and formats but not dtypes.
2. Use the two-sided inequality; do not split a range into two rows.
3. Unsure about a variable? Root. Wrongly in the root costs border width; wrongly in a leaf is a
   wrong answer.
4. Keep your names and pass them to `write_problem`. Debugging an annotation without names is
   miserable.
5. `objconst` if your objective has a constant term.

## 11. Checkpoint

- Why is the capacity limit local to its leaf, although it uses a root column?
- You want a run reproducible in two years. What exactly do you archive?